# 인공지능 작동 과정 · 3변수 선형회귀
## 기온 · 풍속 · 습도로 체감온도 예측하기

---

**인공지능을 위한 선형대수학** · 9장 「최소제곱법과 정규방정식」 응용 · **수업용**

### 이전 수업과 달라진 점

| | 전 (1변수) | 이번 (3변수) |
|---|---|---|
| 입력 | 기온 하나 | **기온 · 풍속 · 습도** |
| 모델 | `y = ax + b` (직선) | **`b = A x`** (행렬) |
| 푸는 법 | `np.polyfit` (내부는 블랙박스) | **정규방정식** `AᵀA x̂ = Aᵀb` |
| 검증 | R² 만 | R² + **직교 조건** + **랭크** |

> 💡 **핵심** — `polyfit` 이 안에서 하던 일을 이제 **우리 손으로** 합니다.


## 진행 순서

1. 데이터 수집과 가공 → 행렬 만들기
2. 인공지능 모델 생성 → 정규방정식으로 풀기
3. 모델 평가 → R² · 직교 조건 · 랭크
4. 정리 및 배포 → 예측 프로그램


---

# 1. 데이터 수집과 데이터 가공


기상 관측 **20회** 기록입니다. 기온·풍속·습도를 재고, 그때의 체감온도를 함께 적었습니다.

| 순번 | 기온(℃) | 풍속(km/h) | 습도(%) | 체감온도(℃) |
|:-:|:-:|:-:|:-:|:-:|
| 1 | 3.3 | 7.6 | 50 | 1.0 |
| 2 | 8.1 | 6.2 | 83 | 7.3 |
| 3 | 6.0 | 17.9 | 63 | 2.4 |
| 4 | −3.9 | 3.1 | 82 | −4.7 |
| 5 | −2.6 | 2.9 | 70 | −3.7 |
| ⋮ | ⋮ | ⋮ | ⋮ | ⋮ |
| 20 | 9.8 | 6.0 | 67 | 9.1 |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

# 기온(℃)
T = np.array([3.3, 8.1, 6.0, -3.9, -2.6, 7.7, -7.9, 6.8, 6.3, 0.4,
              -2.5, -3.0, -3.4, 0.0, 1.1, 2.0, 9.9, 6.3, 3.2, 9.8])
# 풍속(km/h)
V = np.array([7.6, 6.2, 17.9, 3.1, 2.9, 15.4, 14.1, 25.8, 18.4, 15.4,
              14.9, 8.4, 2.3, 7.0, 20.0, 7.2, 11.6, 2.1, 23.6, 6.0])
# 습도(%)
H = np.array([50, 83, 63, 82, 70, 76, 40, 65, 63, 83,
              55, 68, 38, 56, 53, 43, 80, 56, 89, 67.])
# 체감온도(℃) — 예측하려는 값
F = np.array([1.0, 7.3, 2.4, -4.7, -3.7, 4.8, -14.0, 3.1, 3.4, -4.5,
              -7.8, -6.1, -4.3, -2.6, -3.9, 0.4, 8.5, 6.9, -1.8, 9.1])

print('데이터 개수:', len(T))


## 가. 행렬로 정리하기

1변수일 때는 점들을 그냥 늘어놓으면 됐지만, 변수가 3개면 **행렬**이 필요합니다.

$$A\mathbf{x} = \mathbf{b}$$

* `A` — 각 행이 관측 1회, 각 열이 변수 하나
* 맨 오른쪽 **1로 채운 열**은 **절편**(모든 변수가 0일 때의 기본값)


In [ ]:
n = len(T)
A = np.column_stack([T, V, H, np.ones(n)])
b = F

print('A의 크기:', A.shape, '  b의 크기:', b.shape)
print()
print('A의 처음 3행:')
print(A[:3])


> ⚠️ **방정식 20개, 미지수 4개** — 8장에서 배운 **과결정 시스템**입니다.
> 모든 식을 동시에 만족하는 정확한 해는 **없습니다.**


## 나. 훈련 데이터와 테스트 데이터 나누기

모델을 만들 때 쓰는 데이터(**훈련 70%**)와, 잘 맞는지 채점할 데이터(**테스트 30%**)를 나눕니다.

> 💡 **인덱스로 나눕니다.** 값으로 거르면 같은 값이 여러 번 나올 때 짝이 어긋납니다.
> `default_rng(42)` 로 씨앗을 고정해 **매번 같은 결과**가 나오게 합니다.


In [ ]:
rng = np.random.default_rng(42)          # 씨앗 고정 → 항상 같은 분할
idx = rng.permutation(n)
k = int(n * 0.3)

test_idx  = np.sort(idx[:k])
train_idx = np.sort(idx[k:])

A_train, b_train = A[train_idx], b[train_idx]
A_test,  b_test  = A[test_idx],  b[test_idx]

print('훈련 데이터', len(train_idx), '개  |  테스트 데이터', len(test_idx), '개')
print('테스트로 뽑힌 순번:', (test_idx + 1).tolist())


## 다. 산점도로 살펴보기

변수가 3개라 한 그림에 다 넣을 수 없습니다. **변수별로 따로** 그립니다.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
names = ['Temperature', 'Wind speed', 'Humidity']

for j in range(3):
    ax[j].scatter(A_train[:, j], b_train, color='green', s=28, label='train')
    ax[j].scatter(A_test[:, j],  b_test,  color='blue',  s=28, label='test')
    ax[j].set_xlabel(names[j])
    ax[j].set_ylabel('Wind chill')
    ax[j].legend(fontsize=8)

plt.tight_layout(); plt.show()


> 🔎 **발문** — 세 그림 중 어느 것이 가장 뚜렷한 관계를 보이나요?
> 그리고 **거의 관계가 없어 보이는** 변수는 무엇인가요?


---

# 2. 인공지능 모델 생성하기

이전 수업에서는 `np.polyfit` 이 알아서 해 줬습니다. 이번엔 **9장에서 배운 방법**으로 직접 구합니다.

$$A^T A\,\hat{\mathbf{x}} = A^T \mathbf{b}$$


In [ ]:
# 정규방정식 세우기
C = A_train.T @ A_train      # 4 x 4 정사각행렬
d = A_train.T @ b_train

print('A^T A 의 크기:', C.shape, ' ← 데이터가 몇 개든 항상 4x4')
print()
print(C)


In [ ]:
# 풀기
x_hat = np.linalg.solve(C, d)

labels = ['기온', '풍속', '습도', '절편']
for name, coef in zip(labels, x_hat):
    print('%6s : %9.4f' % (name, coef))


### 같은 답인지 확인 — `lstsq`

실무에서 쓰는 함수와 비교합니다. (이전 수업의 `polyfit` 자리)

In [ ]:
x_lstsq, *_ = np.linalg.lstsq(A_train, b_train, rcond=None)

print('정규방정식:', x_hat)
print('lstsq    :', x_lstsq)
print('같은가?', np.allclose(x_hat, x_lstsq))


### 모델을 식으로 쓰면

$$(\text{체감온도}) = c_1(\text{기온}) + c_2(\text{풍속}) + c_3(\text{습도}) + c_4$$


In [ ]:
print('체감온도 = %.3f×기온 %+.3f×풍속 %+.4f×습도 %+.3f'
      % (x_hat[0], x_hat[1], x_hat[2], x_hat[3]))


> 🔎 **발문 — 계수를 문장으로 읽어 봅시다**
>
> * 기온이 1℃ 오르면 체감온도는 몇 도 오르나요?
> * 풍속이 1km/h 늘면? **부호가 왜 음수**인가요?
> * **습도 계수는 어떤가요?** 다른 둘과 비교해 보세요.


---

# 3. 인공지능 모델 평가하기


## 가. 테스트 데이터로 채점

**모델이 한 번도 보지 못한** 데이터로 평가해야 진짜 실력을 알 수 있습니다.

In [ ]:
pred_test = A_test @ x_hat

for i, (p, real) in enumerate(zip(pred_test, b_test), 1):
    print('%2d번  예측 %7.2f   실제 %7.2f   차이 %6.2f' % (i, p, real, real - p))


## 나. 설명력 R²

> ⚠️ **인자 순서 주의** — `r2_score(실제값, 예측값)` 입니다.
> 순서를 바꾸면 **다른 값**이 나옵니다. 흔한 실수입니다.


In [ ]:
from sklearn.metrics import r2_score

R2 = r2_score(b_test, pred_test)          # (실제, 예측) 순서!
print('테스트 R² = %.4f' % R2)
print('평균 절대오차 = %.3f ℃' % np.abs(b_test - pred_test).mean())
print()
print('참고 — 순서를 바꾸면:', round(r2_score(pred_test, b_test), 4), '← 다른 값!')


In [ ]:
# 예측 vs 실제 그림 — 점이 대각선에 가까울수록 좋은 모델
plt.figure(figsize=(4.4, 4.4))
plt.scatter(b_test, pred_test, color='blue', s=40)
lim = [min(b_test.min(), pred_test.min()) - 1, max(b_test.max(), pred_test.max()) + 1]
plt.plot(lim, lim, 'r--', linewidth=1)
plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.title('Predicted vs Actual')
plt.tight_layout(); plt.show()


## 다. 선형대수로 한 번 더 확인

R² 는 “얼마나 잘 맞나”만 알려줍니다. 우리는 **제대로 푼 게 맞는지**도 확인할 수 있습니다.

### ① 직교 조건 — 9장의 핵심

$$A^T(\mathbf{b} - A\hat{\mathbf{x}}) = \mathbf{0}$$


In [ ]:
r = b_train - A_train @ x_hat        # 오차(잔차) 벡터

print('A^T r =', np.round(A_train.T @ r, 8))
print('0에 가까운가?', np.allclose(A_train.T @ r, 0))


> 오차가 `Col A` 에 **수직** — 수선의 발을 제대로 찾았다는 증거입니다.

### ② 랭크 — 중복된 변수가 있나


In [ ]:
print('rank(A) =', np.linalg.matrix_rank(A_train), ' / 열 개수', A_train.shape[1])


> 같으면 **중복 없음**. 만약 “기온(℃)”과 “기온(℉)”을 둘 다 넣었다면
> 랭크가 모자라고 `A^T A` 의 역행렬이 없어집니다. (5장·9장)

### ③ 습도는 정말 필요한가

계수가 거의 0이었습니다. **빼고 다시 만들어** 비교해 봅시다.

In [ ]:
A2_train = A_train[:, [0, 1, 3]]      # 습도 열 제거
A2_test  = A_test[:,  [0, 1, 3]]

x2, *_ = np.linalg.lstsq(A2_train, b_train, rcond=None)
R2_2 = r2_score(b_test, A2_test @ x2)

print('습도 포함 R² = %.4f' % R2)
print('습도 제외 R² = %.4f' % R2_2)
print()
print('변수를 줄였는데 결과가 어떻게 되었나요?')


> 🔎 **가장 중요한 장면**
>
> 변수를 **뺐는데 테스트 성적이 오히려 올랐습니다.**
>
> 습도 계수(약 0.02)는 훈련 데이터의 **우연한 흔들림**을 따라간 것입니다.
> 훈련 데이터에는 조금 더 잘 맞지만, **처음 보는 데이터에서는 방해**가 됩니다.
> 이것이 **과적합**(overfitting)입니다.
>
> **발문 —** 그럼 변수는 많을수록 좋은 걸까요?


> 💡 **한 걸음 더** — 습도가 체감온도와 정말 무관할까요?
>
> 겨울철 체감온도 공식에는 습도가 들어가지 않지만, 여름철 불쾌지수에는 들어갑니다.
> **데이터가 말해 주는 것**과 **세상의 진실**은 다를 수 있습니다.
> 우리가 겨울 데이터만 모았기 때문에 나온 결론입니다.


---

# 4. 정리 및 배포

## 체감온도 예측 프로그램

기온·풍속·습도를 입력하면 체감온도를 예측합니다.

In [ ]:
print('-' * 18, '체감온도 예측', '-' * 18)

t = float(input('기온을 입력하세요 (℃): '))
v = float(input('풍속을 입력하세요 (km/h): '))
h = float(input('습도를 입력하세요 (%): '))

new = np.array([t, v, h, 1.0])            # 맨 뒤 1은 절편
print()
print('예측 체감온도: %.2f ℃' % (new @ x_hat))


---

## 오늘 배운 것

| 단계 | 이전 수업 | 이번 수업 |
|---|---|---|
| 데이터 | 점들의 나열 | **행렬 `A`, 벡터 `b`** |
| 모델 | `polyfit` | **정규방정식 `AᵀA x̂ = Aᵀb`** |
| 평가 | R² | R² + **직교 조건** + **랭크** |
| 해석 | 기울기 하나 | **변수별 기여도 비교** |

### 이어지는 활동

각자 **최종 프로젝트**에서 자기 주제로 같은 분석을 합니다.
데이터는 **엑셀 양식**에 정리해 업로드하면 됩니다.
